<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

# 11_03 Transformer decoder

In this notebook, you will implement the core components of the Transformer decoder architecture, step by step, from scratch using PyTorch.    


In [12]:
import math
import torch
import torch.nn as nn
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.shape[1], :]


class LearnedPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)
        nn.init.normal_(self.pos_embedding.weight, mean=0.0, std=0.02)

    def forward(self, x):
        B, T, _ = x.shape
        positions = torch.arange(0, T, device=x.device)
        return x + self.pos_embedding(positions)


class AddNorm(nn.Module):
    def __init__(self, d_model, pre_norm):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pre_norm = pre_norm

    def forward(self, x, sublayer):
        if self.pre_norm:
            return x + sublayer(self.norm(x))
        else:
            return self.norm(x + sublayer(x))


class FFN(nn.Module):
    def __init__(self, d_model, d_ff, activation="gelu"):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

        if activation.lower() == "relu":
            self.activation = nn.ReLU()
        elif activation.lower() == "gelu":
            self.activation = nn.GELU()
        else:
            raise ValueError(f"Unsupported activation: {activation}")

    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))

### 1. `MultiHeadAttention` 

You will implement a **generic multi-head attention** module that can act as:
- Cross-attention: `x_q` (decoder states), `x_kv` (encoder outputs)
- Self-attention: `x_q == x_kv`


**Steps**
1. In `__init__`:
   - Create three `nn.Linear` layers for Q, K, and V projections.  
   - Create one output linear layer `W_o` for combining heads.
   - Store useful attributes (`num_heads`, `d_k`, `d_v`).
   - Define the scale factor `self.scale`.

2. In `forward(x_q, x_kv, attn_mask=None)`:
    - Input / Output:
        - `x_q`: `(B, T_q, d_model)` — queries come from this sequence
        - `x_kv`: `(B, T_k, d_model)` — keys/values come from this sequence
        - `attn_mask` (optional, additive): broadcastable to `(B, h, T_q, T_k)`, with `0` for keep and `-inf` to mask
        - Return shape: `(B, T_q, d_model)`
   - Compute Q, K, V = linear projections of `x`.
   - Reshape them to `(B, num_heads, T_q, d_k)`, `(B, num_heads, T_k, d_k)` and `(B, num_heads, T_k, d_v)`.
   - Scores:
     $$
     \text{scores} = \frac{Q K^\top}{\sqrt{d_k}} \quad\Rightarrow\quad (B, h, T_q, T_k)
     $$
   - If `attn_mask` is given, **add** it to scores (additive mask).
   - Apply softmax over the last dimension to get attention weights.
   - Compute attention output = softmax(scores) × V.
   - Concatenate all heads and apply `W_o`.

3. Return the final output tensor `(B, T_q, d_model)`.


In [13]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.d_k = d_k
        self.d_v = d_v
        self.d_model = d_model

        self.W_Q = nn.Linear(d_model, num_heads * d_k, bias=False)
        self.W_K = nn.Linear(d_model, num_heads * d_k, bias=False)
        self.W_V = nn.Linear(d_model, num_heads * d_v, bias=False)

        self.W_o = nn.Linear(num_heads * d_v, d_model, bias=False)

        self.scale = math.sqrt(d_k)

    def forward(self, x_q, x_kv, attn_mask = None):
        
        B, T_q, d_model = x_q.shape        # x_q : (B, T_q, d_model)
        B, T_k, d_model= x_kv.shape        # x_kv: (B, T_k, d_model)


        Q_all = self.W_Q(x_q)
        K_all = self.W_K(x_kv) 
        V_all = self.W_V(x_kv)  

        Q = Q_all.view(B, T_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K_all.view(B, T_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V_all.view(B, T_k, self.num_heads, self.d_v).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / self.scale

        if attn_mask is not None:
            scores = scores + attn_mask

        attention_weights = torch.softmax(scores, dim=-1)

        out_heads = attention_weights @ V
        out_joined = out_heads.permute(0, 2, 1, 3).flatten(start_dim=2)

        output = self.W_o(out_joined)  # (B, T_q, d_model)

        return output

In [14]:
# Parámetros de prueba
B, T_q, T_k, d_model = 2, 10, 15, 64
d_k, d_v, num_heads = 16, 16, 4

attn = MultiHeadAttention(d_model=d_model, d_k=d_k, d_v=d_v, num_heads=num_heads)

# Tensores de prueba
x_decoder = torch.randn(B, T_q, d_model) # Queries del Decoder
x_encoder = torch.randn(B, T_k, d_model) # Keys/Values del Encoder

# 1. Prueba Cross-Attention (x_q != x_kv)
out_cross = attn(x_q=x_decoder, x_kv=x_encoder)

# 2. Prueba Self-Attention (x_q == x_kv)
out_self = attn(x_q=x_decoder, x_kv=x_decoder)

print("Cross-Attention Output Shape:", out_cross.shape) # Esperado: (2, 10, 64)
print("Self-Attention Output Shape :", out_self.shape)  # Esperado: (2, 10, 64)

Cross-Attention Output Shape: torch.Size([2, 10, 64])
Self-Attention Output Shape : torch.Size([2, 10, 64])


### 2. `DecoderLayer`

Build one Transformer decoder layer with three sublayers:
1) Masked self-attention over the decoder states  
2) Cross-attention (queries from decoder, keys/values from encoder outputs)  
3) Positionwise feedforward network
Each wrapped with Add & Norm (residual + LayerNorm).



**Steps**
1. In `__init__`:
   - Two attention modules (can reuse the generic `MultiHeadAttention`):
     - `self_attn` for masked self-attention
     - `cross_attn` for encoder–decoder attention
   - `feed_forward`: `FFN(d_model, d_ff, ...)`
   - Three `AddNorm` blocks: for self-attn, cross-attn, and ffn sublayers
   - Support pre-norm or post-norm behavior

2. In `forward(x, enc_out, self_mask=None, cross_mask=None)`:
   - Input / Output:
        - `x`: `(B, T_dec, d_model)`, decoder hidden states
        - `enc_out`: `(B, T_enc, d_model)`, encoder outputs (memory)
        - `self_mask`: `(B, h, T_dec, T_dec)`, causal + (optional) target padding mask (additive: 0 keep, -inf mask)
        - `cross_mask`: `(B, h, T_dec, T_enc)`, (optional) encoder padding mask as seen from decoder
        - Return shape: `(B, T_dec, d_model)` 
   - Self-attn block (use `self_mask`):
     - (pre-norm) norm → self_attn → residual  
       or (post-norm) self_attn → residual → norm
   - Cross-attn block (use `cross_mask`):
     - same Add&Norm pattern, with queries from the decoder and keys and values from the encoder
   - FFN block:
     - same Add&Norm pattern with `feed_forward`

**Checks**
- All residual paths preserve `(B, T_dec, d_model)`.
- Masks are **additive** and broadcast to `(B, h, T_q, T_k)`.
- Causal constraint is enforced via `self_mask` (upper triangle masked).


In [15]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_k, d_v, num_heads, d_ff, prenorm, activation="gelu"):
        super().__init__()

        # Subcapas de atención y FFN
        self.self_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, d_k, d_v, num_heads)
        self.ffn = FFN(d_model, d_ff, activation)

        # Tres bloques AddNorm (uno por subcapa)
        self.add_norm_1 = AddNorm(d_model, pre_norm=prenorm)
        self.add_norm_2 = AddNorm(d_model, pre_norm=prenorm)
        self.add_norm_3 = AddNorm(d_model, pre_norm=prenorm)

    def forward(self, x, enc_out, self_mask=None, cross_mask=None):
        # 1. Self-Attention con máscara causal
        x = self.add_norm_1(
            x, lambda x_: self.self_attn(x_q=x_, x_kv=x_, attn_mask=self_mask)
        )

        # 2. Cross-Attention (Queries de x, Keys y Values de enc_out)
        x = self.add_norm_2(
            x, lambda x_: self.cross_attn(x_q=x_, x_kv=enc_out, attn_mask=cross_mask)
        )

        # 3. Feed Forward Network
        x = self.add_norm_3(x, self.ffn)

        return x

### 3. Mask helper functions (additive masks)

You will implement **additive masks** that are added to attention score tensors before softmax.  
Convention: `0` = keep, `-inf` = mask out.

* Causal mask for decoder self-attention  
    - Shape: `(B, h, T_dec, T_dec)`  
    - Upper triangular (strictly above diagonal) entries should be `-inf` (masked); diagonal and below are 0.  
    - Repeats the same triangular pattern for every batch element and head.

**Signature**
```python
def make_causal_mask(B: int, num_heads: int, T_dec: int, device=None) -> torch.Tensor:
    ...


In [16]:
def make_causal_mask(B: int, num_heads: int, T_dec: int, device=None) -> torch.Tensor:
    mask = torch.triu(torch.full((T_dec, T_dec), float('-inf'), device=device), diagonal=1)
    causal_mask = mask.unsqueeze(0).unsqueeze(0).expand(B, num_heads, T_dec, T_dec)
    
    return causal_mask


### 4. Full `Decoder` stack (batched)

Assemble a complete **Transformer decoder**:
- Positional encoding (sinusoidal or learned) added to inputs.
- N stacked `DecoderLayer`s (`nn.ModuleList`).
- Optional final LayerNorm (it is often used for pre-norm setups).

**Inputs**
- `x_dec`: `(B, T_dec, d_model)`, decoder token embeddings
- `enc_out`: `(B, T_enc, d_model)`, encoder outputs (memory)
- Return: contextual vectors `(B, T_dec, d_model)`, we will include the decoder head (Linear + softmax) later on.

**Behavior**
1. Add positional encodings to `x_dec`.
2. Build a **causal self-attention mask**: `(B, h, T_dec, T_dec)`.
3. Pass through `num_layers` decoder layers using these masks.
4. Optionally apply a final `LayerNorm`.

**Checks**
- All layers preserve `(B, T_dec, d_model)`.


In [17]:
class Decoder(nn.Module):
    def __init__(
        self, 
        d_model, 
        d_k, 
        d_v, 
        num_heads, 
        num_layers, 
        d_ff, 
        max_len, 
        prenorm=True, 
        activation="gelu", 
        positional_encoding_type="sinusoidal"
    ):
        super().__init__()

        # Selección de la codificación posicional
        if positional_encoding_type == "sinusoidal":
            self.pos_encoder = SinusoidalPositionalEncoding(max_len=max_len, d_model=d_model)
        elif positional_encoding_type == "learned":
            self.pos_encoder = LearnedPositionalEncoding(max_len=max_len, d_model=d_model)
        else:
            raise ValueError("positional_encoding_type debe ser 'sinusoidal' o 'learned'")

        # Pila de capas del Decoder
        self.layers = nn.ModuleList([
            DecoderLayer(
                d_model=d_model,
                d_k=d_k,
                d_v=d_v,
                num_heads=num_heads,
                d_ff=d_ff,
                prenorm=prenorm,
                activation=activation
            ) for _ in range(num_layers)
        ])

        self.prenorm = prenorm
        if self.prenorm:
            self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x_dec, enc_out, self_mask=None, cross_mask=None):
        B, T_dec, _ = x_dec.shape

        # 1. Sumar embeddings posicionales
        x = self.pos_encoder(x_dec)

        # 2. Generar máscara causal si no se provee una
        if self_mask is None:
            self_mask = make_causal_mask(B, self.layers[0].self_attn.num_heads, T_dec, device=x.device)

        # 3. Pasar secuencialmente por las N capas
        for layer in self.layers:
            x = layer(x, enc_out, self_mask=self_mask, cross_mask=cross_mask)

        # 4. LayerNorm final opcional para configuraciones Pre-Norm
        if self.prenorm:
            x = self.final_norm(x)

        return x

### 5. `DecoderHead` (Linear → Vocab)

The decoder outputs contextual vectors `(B, T, d_model)`. We must turn them into **vocabulary logits** `(B, T, V)`.

**Tasks**
1. In `__init__(...)`:
   - Create a projection to vocabulary size `V`. 
2. In `forward(h_dec)`:
   - Input: `(B, T, d_model)`  
   - Output: `(B, T, vocab_size)` logits (no softmax).

**Note**
- Keep a bias term (optional).


In [18]:
class DecoderHead(nn.Module):
    def __init__(self, d_model, vocab_size, bias=True):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab_size, bias=bias)

    def forward(self, h_dec):
        # h_dec: (B, T, d_model) -> Salida: (B, T, vocab_size)
        return self.proj(h_dec)

In [ ]:
# Configuración
B, T_dec, T_enc = 2, 10, 15
d_model = 64
num_heads = 4
d_k = d_v = d_model // num_heads  # 16
d_ff = 256
num_layers = 2
vocab_size = 1000

# Módulos
decoder = Decoder(
    d_model=d_model,
    d_k=d_k,
    d_v=d_v,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_len=512,
    prenorm=True,
    activation="gelu",
    positional_encoding_type="sinusoidal"
)
head = DecoderHead(d_model=d_model, vocab_size=vocab_size)

# Tensores ficticios
x_dec = torch.randn(B, T_dec, d_model)   # Salida del target embedding
enc_out = torch.randn(B, T_enc, d_model) # Salida del Encoder

# Forward pass
with torch.no_grad():
    h_dec = decoder(x_dec, enc_out)
    logits = head(h_dec)

print("✅ Decoder Output shape:", h_dec.shape)  # Esperado: (2, 10, 64)
print("✅ Logits Output shape :", logits.shape) # Esperado: (2, 10, 1000)


✅ Decoder Output shape: torch.Size([2, 10, 64])
✅ Logits Output shape : torch.Size([2, 10, 1000])
